# Batch Tournament Product Evidence Harness

End-to-end batch workflow for the **tournament-first** Product Evidence Harness.

Primary batch flow:

```text
CSV/XLSX input
  → tournament discovery per row, capped at 4 SerpAPI credits
  → candidate pool + batch scraping
  → champion URL + runner-up analysis
  → production URL gate
  → final_submission.csv + review_queue.csv + row artifacts
```

Production handoff rule:

```text
production_url_ready == True
production_url_status == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL"
needs_review == False
```

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

PROJECT_ROOT

In [ ]:
from product_evidence_harness import HarnessConfig

config = HarnessConfig.from_env(PROJECT_ROOT / ".env")
assert config.tournament.enabled is True, "Tournament mode should be enabled for the primary batch flow."
assert config.tournament.max_serp_credits <= 4, "Tournament mode must respect the 4-credit SerpAPI cap."

pd.Series({
    "tournament_enabled": config.tournament.enabled,
    "max_serp_credits": config.tournament.max_serp_credits,
    "candidate_pool": config.tournament.candidate_pool,
    "preflight_top_k": config.tournament.preflight_top_k,
    "batch_size": config.tournament.batch_size,
    "max_batches": config.tournament.max_batches,
    "scrape_concurrency": config.scrape_concurrency,
}).to_frame("value")

In [ ]:
input_path = PROJECT_ROOT / "data" / "products.csv"
output_path = PROJECT_ROOT / "outputs" / "final_submission.csv"
workers = 4

print("Input:", input_path)
print("Output:", output_path)

In [ ]:
# Preview input while preserving EAN/GTIN as text.
# Required columns: row_id, main_text, country_code
# Optional columns: ean, retailer_name, language_code, region
if input_path.exists():
    display(pd.read_csv(input_path, dtype=str).head())
else:
    print("Input file not found. Update input_path above.")

In [ ]:
cmd = [
    sys.executable,
    "batch_main.py",
    "--input", str(input_path),
    "--output", str(output_path),
    "--workers", str(workers),
]
print("Command:", " ".join(cmd))
print("Tournament mode is primary/default through .env and HarnessConfig.")

# Uncomment to run from notebook. For long batches, terminal execution is easier to monitor.
# subprocess.run(cmd, cwd=PROJECT_ROOT, check=True)

In [ ]:
if output_path.exists():
    df = pd.read_csv(output_path, dtype=str)
    print("Rows:", len(df))
    display(df.head())
else:
    df = pd.DataFrame()
    print("Run batch_main.py first to create:", output_path)

In [ ]:
handoff_cols = [
    "row_id", "main_text", "country_code", "retailer_name",
    "product_url", "production_url_ready", "production_url_status",
    "browser_openable", "highly_scrapable", "exact_product_url_match",
    "needs_review", "verified_exact_url", "quality_tier",
    "coding_readiness_status", "failure_taxonomy",
    "production_url_reasons", "product_coding_input_path",
]

if not df.empty:
    display(df[[c for c in handoff_cols if c in df.columns]].head(25))

In [ ]:
if not df.empty:
    ready = df[
        (df.get("production_url_ready", "").astype(str).str.lower().isin(["true", "1", "yes"]))
        & (df.get("production_url_status", "") == "PRODUCTION_READY_EXACT_SCRAPABLE_BROWSER_URL")
        & (~df.get("needs_review", "true").astype(str).str.lower().isin(["true", "1", "yes"]))
    ].copy()
    review_only = df[~df.index.isin(ready.index)].copy()
    print("Production-ready handoff rows:", len(ready), "/", len(df))
    print("Review-only rows:", len(review_only))
    display(ready[[c for c in handoff_cols if c in ready.columns]].head(50))
else:
    ready = pd.DataFrame()
    review_only = pd.DataFrame()

In [ ]:
if not review_only.empty:
    display(review_only[[c for c in handoff_cols if c in review_only.columns]].head(50))

In [ ]:
metrics_path = PROJECT_ROOT / "outputs" / "metrics.json"
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text(encoding="utf-8"))
    print(json.dumps(metrics, indent=2, ensure_ascii=False))
else:
    print("metrics.json not found:", metrics_path)

In [ ]:
# Build a tournament summary table from row artifacts.
rows = []
if not df.empty:
    for _, rec in df.head(50).iterrows():
        row_id = str(rec.get("row_id", ""))
        bracket_path = PROJECT_ROOT / "output" / row_id / "tournament_bracket.json"
        if bracket_path.exists():
            bracket = json.loads(bracket_path.read_text(encoding="utf-8"))
            rows.append({
                "row_id": row_id,
                "product_url": rec.get("product_url"),
                "champion_url": bracket.get("champion_url"),
                "runner_up_url": bracket.get("runner_up_url"),
                "champion_margin": bracket.get("champion_margin"),
                "search_credits_used": bracket.get("search_credits_used"),
                "search_credit_limit": bracket.get("search_credit_limit"),
                "raw_candidate_count": bracket.get("raw_candidate_count"),
                "scraped_candidate_count": bracket.get("scraped_candidate_count"),
                "champion_status": bracket.get("champion_status"),
            })

tournament_df = pd.DataFrame(rows)
display(tournament_df)

In [ ]:
# Inspect one row artifact packet end to end.
if not df.empty:
    row_id = str(df.iloc[0]["row_id"])
    row_dir = PROJECT_ROOT / "output" / row_id
    print("Row artifact directory:", row_dir)
    expected = [
        "final_row.csv",
        "tournament_bracket.md",
        "tournament_bracket.json",
        "batch_winners.csv",
        "quality_assessment.md",
        "product_coding_input.json",
        "evidence_graph.json",
    ]
    if row_dir.exists():
        existing = {p.name for p in row_dir.iterdir()}
        display(pd.DataFrame({"artifact": expected, "exists": [name in existing for name in expected]}))
    else:
        print("Row directory not found.")

In [ ]:
if not df.empty:
    row_id = str(df.iloc[0]["row_id"])
    row_dir = PROJECT_ROOT / "output" / row_id
    winners_path = row_dir / "batch_winners.csv"
    if winners_path.exists():
        display(pd.read_csv(winners_path, dtype=str))
    else:
        print("batch_winners.csv not found for selected row")

In [ ]:
if not df.empty:
    row_id = str(df.iloc[0]["row_id"])
    row_dir = PROJECT_ROOT / "output" / row_id
    coding_path = row_dir / "product_coding_input.json"
    if coding_path.exists():
        payload = json.loads(coding_path.read_text(encoding="utf-8"))
        print(json.dumps({
            "selected_url": payload.get("selected_url"),
            "verified_exact_url": payload.get("verified_exact_url"),
            "quality_tier": payload.get("quality_tier"),
            "coding_readiness_status": payload.get("coding_readiness_status"),
            "review_flags": payload.get("review_flags"),
        }, indent=2, ensure_ascii=False))
    else:
        print("product_coding_input.json not found for selected row")